# FuDGE / FF1 — End-to-End Demo

Replication of: *"Automatic Evaluation of Task-Oriented Dialogue Flows"* (arxiv 2411.10416)

This notebook walks through:
1. Building a `DialogueFlow` DAG
2. Computing **FuDGE** (Fuzzy Dialogue-Graph Edit Distance)
3. Computing **FF1** (Flow-F1)
4. Loading the STAR dataset
5. Running a quick discrimination sanity check

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
from src.graph import DialogueFlow, flow_from_intent_sequence
from src.fudge import fudge, fudge_naive, avg_fudge
from src.ff1 import ff1, ff1_breakdown
from src.embeddings import cosine_dist, encode

## 1. Build a simple DialogueFlow

In [ ]:
bank_flow = flow_from_intent_sequence(
    name="banking",
    intents=[
        {"id": "greet_user",      "actor": "user",  "utterances": [
            "Hello, I need help with my account.",
            "Hi there, I have a banking question.",
        ]},
        {"id": "greet_agent",     "actor": "agent", "utterances": [
            "Hello! How can I help you today?",
            "Hi! What can I do for you?",
        ]},
        {"id": "request_balance", "actor": "user",  "utterances": [
            "What is my current balance?",
            "Can you check my account balance?",
        ]},
        {"id": "provide_balance", "actor": "agent", "utterances": [
            "Your current balance is $1,234.56.",
            "You have $567.89 in your account.",
        ]},
        {"id": "farewell",        "actor": "user",  "utterances": [
            "Thank you, goodbye.",
            "That's all I needed. Bye!",
        ]},
    ]
)

print(bank_flow)
print("Nodes:", bank_flow.intent_nodes())
print("Source:", bank_flow.source_nodes())
print("Sink:",   bank_flow.sink_nodes())

## 2. Compute FuDGE for a matching dialogue

In [ ]:
# A dialogue that closely follows the flow
matching_dialogue = [
    ("user",  "Hi, I need help with my bank account please."),
    ("agent", "Hello! How can I assist you today?"),
    ("user",  "What's my balance?"),
    ("agent", "Your current balance is $1,234.56."),
    ("user",  "Thank you very much. Goodbye!"),
]

# A dialogue from a completely different domain
mismatch_dialogue = [
    ("user",  "I want to book a hotel room for the weekend."),
    ("agent", "Of course! What dates are you looking at?"),
    ("user",  "Friday to Sunday please."),
    ("agent", "Great, we have availability. Shall I book?"),
    ("user",  "Yes please. Thanks!"),
]

score_in  = fudge(matching_dialogue,  bank_flow, variant="min")
score_out = fudge(mismatch_dialogue,  bank_flow, variant="min")

print(f"FuDGE (in-task):     {score_in:.4f}")
print(f"FuDGE (out-of-task): {score_out:.4f}")
print(f"\nSanity check (in < out): {score_in < score_out}")

## 3. Naive vs Efficient FuDGE (consistency check)

In [ ]:
score_naive    = fudge_naive(matching_dialogue, bank_flow, variant="min")
score_efficient = fudge(matching_dialogue, bank_flow, variant="min")

print(f"Naive FuDGE:     {score_naive:.4f}")
print(f"Efficient FuDGE: {score_efficient:.4f}")
print(f"Match (tol=1e-3): {abs(score_naive - score_efficient) < 1e-3}")

## 4. Compute FF1

In [ ]:
dialogues = [matching_dialogue] * 10  # simulate 10 in-task dialogues

bd = ff1_breakdown(dialogues, bank_flow)
for key, val in bd.items():
    print(f"  {key:25s}: {val:.4f}" if isinstance(val, float) else f"  {key:25s}: {val}")

## 5. Load STAR Dataset

In [ ]:
from src.data_loader import load_star

data = load_star(domains=["banking", "hotel"], max_dialogues_per_domain=50)

for domain, info in data.items():
    print(f"\nDomain: {domain}")
    print(f"  Flow: {info['flow']}")
    print(f"  Total dialogues: {len(info['dialogues'])}")
    if info['dialogues']:
        d = info['dialogues'][0]
        print(f"  First dialogue (first 3 turns):")
        for actor, utt in d[:3]:
            print(f"    [{actor}] {utt[:80]}")

## 6. Quick Discrimination Check

In [ ]:
from src.fudge import fudge
import numpy as np

for domain, info in data.items():
    flow = info["flow"]
    in_diags  = info["split"]["in_task"][:20]
    out_diags = info["split"]["out_of_task"][:20]

    in_scores  = [fudge(d, flow) for d in in_diags  if d]
    out_scores = [fudge(d, flow) for d in out_diags if d]

    print(f"\n{domain}:")
    print(f"  In-task  FuDGE: {np.mean(in_scores):.3f} ± {np.std(in_scores):.3f}")
    print(f"  Out-task FuDGE: {np.mean(out_scores):.3f} ± {np.std(out_scores):.3f}")
    print(f"  Separation:     {np.mean(in_scores) < np.mean(out_scores)}")

## 7. Visualise Flow DAG

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

G = bank_flow.graph
real_nodes = bank_flow.intent_nodes()
pos = nx.spring_layout(G, seed=42)

colors = []
for n in G.nodes():
    actor = G.nodes[n].get("actor")
    if actor == "user":
        colors.append("lightblue")
    elif actor == "agent":
        colors.append("lightgreen")
    else:
        colors.append("lightgrey")

labels = {n: G.nodes[n].get("name", n) for n in G.nodes()}

plt.figure(figsize=(10, 6))
nx.draw(G, pos, labels=labels, node_color=colors,
        node_size=2000, font_size=9, arrows=True,
        edge_color="grey", arrowsize=20)
plt.title("Banking Flow DAG", fontsize=14)
plt.tight_layout()
plt.show()